# EDA — Bank Credit Card Transactions

Exploratory analysis of `creditcard.csv`: PCA features V1–V28, Amount, Time, and class imbalance.

**Place** `creditcard.csv` **in** `data/raw/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.data_loader import load_creditcard
from src.preprocessing import class_distribution, clean_creditcard_data

sns.set_theme(style="whitegrid")
RAW = ROOT / "data" / "raw"

In [ ]:
raw = load_creditcard(raw_dir=RAW)
print("Raw shape:", raw.shape)
print("Missing:\n", raw.isna().sum().sum())
print("Duplicates:", raw.duplicated().sum())
cc = clean_creditcard_data(raw)
print("Cleaned shape:", cc.shape)
cc.head()

## Class imbalance

In [ ]:
dist = class_distribution(cc["Class"], label="Class")
display(dist)
fraud_rate = cc["Class"].mean()
print(f"Fraud rate: {fraud_rate:.4%} ({cc['Class'].sum()} fraud / {len(cc)} total)")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=cc, x="Class", ax=ax, palette="Set1")
ax.set_title("Credit-card class imbalance")
plt.show()

## Amount and Time vs Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=cc, x="Class", y="Amount", ax=axes[0], showfliers=False)
axes[0].set_title("Amount by Class (outliers hidden)")
sns.histplot(
    data=cc.sample(min(len(cc), 50000), random_state=42),
    x="Time",
    hue="Class",
    ax=axes[1],
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
)
axes[1].set_title("Time density by Class (sample)")
plt.tight_layout()
plt.show()

print(cc.groupby("Class")["Amount"].describe())

## PCA feature snapshot

In [ ]:
v_cols = [c for c in cc.columns if c.startswith("V")]
corr_with_class = cc[v_cols + ["Class"]].corr()["Class"].drop("Class").abs().sort_values(ascending=False)
print("Top |corr| with Class:")
display(corr_with_class.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
corr_with_class.head(10).plot(kind="bar", ax=ax, color="teal")
ax.set_title("Top PCA features by |correlation| with Class")
plt.tight_layout()
plt.show()

## Takeaways

- Extremely imbalanced (~0.17% fraud typical for this dataset).
- Features are already PCA-anonymized — no further geo/behavioral engineering.
- Scale `Amount`/`Time` before modeling; use SMOTE on train only; evaluate with AUC-PR.